In [1]:
!pip install -q -U haystack-ai sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 17.1 MB/s eta 0:00:00


In [2]:
import sys
import pandas as pd
import numpy as np
import haystack

from pathlib import Path

from haystack import Pipeline, Document
from haystack.components.converters import PyPDFToDocument

from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever
)

from haystack.document_stores.in_memory import InMemoryDocumentStore

from sentence_transformers import SentenceTransformer

print("Python:", sys.version)
print("Haystack:", haystack.__version__)
print("✓ Imports successful")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Haystack: 3.1.1
✓ Imports successful


In [4]:

from google.colab import files

uploaded = files.upload()

pdf_files = [
    file for file in uploaded.keys()
    if file.lower().endswith(".pdf")
]

print("\nUploaded PDF files:")

for file in pdf_files:
    print("-", file)

print("\nNumber of PDFs:", len(pdf_files))


Saving week7pdf5.pdf to week7pdf5 (1).pdf
Saving week7pdf4.pdf to week7pdf4.pdf
Saving week7pdf3.pdf to week7pdf3.pdf
Saving week7pdf2.pdf to week7pdf2.pdf
Saving week7pdf1.pdf to week7pdf1.pdf

Uploaded PDF files:
- week7pdf5 (1).pdf
- week7pdf4.pdf
- week7pdf3.pdf
- week7pdf2.pdf
- week7pdf1.pdf

Number of PDFs: 5


In [5]:
pdf_paths = [Path(file) for file in pdf_files]

if len(pdf_paths) != 5:
    raise ValueError(
        f"Expected exactly 5 PDFs, but found {len(pdf_paths)}."
    )

print("✓ Exactly 5 PDF documents found")

for i, path in enumerate(pdf_paths, start=1):
    print(f"{i}. {path.name}")

✓ Exactly 5 PDF documents found
1. week7pdf5 (1).pdf
2. week7pdf4.pdf
3. week7pdf3.pdf
4. week7pdf2.pdf
5. week7pdf1.pdf


In [6]:
converter = PyPDFToDocument()

conversion_result = converter.run(
    sources=pdf_paths
)

documents = conversion_result["documents"]

print("✓ PDF conversion completed")
print("Total Haystack documents:", len(documents))

✓ PDF conversion completed
Total Haystack documents: 5


In [7]:
for i, doc in enumerate(documents, start=1):

    print("=" * 70)
    print(f"Document {i}")
    print("Characters:", len(doc.content))
    print("Metadata:", doc.meta)

    print("\nPreview:")
    print(doc.content[:300].replace("\n", " "))

Document 1
Characters: 44162
Metadata: {'file_path': 'week7pdf5 (1).pdf'}

Preview:
Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks Nils Reimers and Iryna Gurevych Ubiquitous Knowledge Processing Lab (UKP-TUDA) Department of Computer Science, Technische Universit¨at Darmstadt www.ukp.tu-darmstadt.de Abstract BERT (Devlin et al., 2018) and RoBERTa (Liu et al., 2019) h
Document 2
Characters: 236748
Metadata: {'file_path': 'week7pdf4.pdf'}

Preview:
Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan Rewon Child Aditya Ramesh Daniel M. Ziegler Je
Document 3
Characters: 69076
Metadata: {'file_path': 'week7pdf3.pdf'}

Preview:
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, 

In [8]:
bm25_store = InMemoryDocumentStore()

print("✓ BM25 DocumentStore created")

✓ BM25 DocumentStore created


In [9]:
bm25_store.write_documents(documents)

print("✓ Documents indexed")
print("Total documents:", bm25_store.count_documents())

✓ Documents indexed
Total documents: 5


In [10]:
bm25_retriever = InMemoryBM25Retriever(
    document_store=bm25_store,
    top_k=3
)

print("✓ BM25 Retriever created")

✓ BM25 Retriever created


In [11]:
bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    "retriever",
    bm25_retriever
)

print("✓ BM25 pipeline created")

✓ BM25 pipeline created


In [12]:
questions = [
    "What is the main objective of the research?",
    "What problem does the research attempt to solve?",
    "What methodology is used in the study?",
    "What dataset is used?",
    "What model or algorithm is proposed?",
    "How is the proposed method evaluated?",
    "What are the main findings of the study?",
    "What are the key experimental results?",
    "What are the limitations of the proposed approach?",
    "What future work is suggested?"
]

print("Total questions:", len(questions))

Total questions: 10


In [13]:
bm25_results = []

for i, question in enumerate(questions, start=1):

    result = bm25_pipeline.run({
        "retriever": {
            "query": question
        }
    })

    retrieved_docs = result["retriever"]["documents"]

    bm25_results.append({
        "question": question,
        "documents": retrieved_docs
    })

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print(question)

    for rank, doc in enumerate(retrieved_docs, start=1):

        print(f"\nRank {rank}")
        print("Score:", doc.score)
        print(
            "Text:",
            doc.content[:300].replace("\n", " ")
        )


QUESTION 1
What is the main objective of the research?

Rank 1
Score: 3.1192958533739996
Text: Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan Rewon Child Aditya Ramesh Daniel M. Ziegler Je

Rank 2
Score: 2.989259051428672
Text: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI Research;‡University C

Rank 3
Score: 2.679015066436271
Text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google B

In [14]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✓ Sentence Transformer loaded")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Sentence Transformer loaded
Embedding dimension: 384


/tmp/ipykernel_3380/3335318516.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [15]:
document_texts = [
    doc.content for doc in documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("✓ Document embeddings generated")
print("Shape:", document_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Document embeddings generated
Shape: (5, 384)


In [16]:
embedded_documents = []

for doc, embedding in zip(
    documents,
    document_embeddings
):

    embedded_doc = Document(
        content=doc.content,
        meta=doc.meta,
        embedding=embedding.tolist()
    )

    embedded_documents.append(embedded_doc)

print("✓ Embeddings attached to documents")

✓ Embeddings attached to documents


In [17]:
dense_store = InMemoryDocumentStore(
    embedding_similarity_function="cosine"
)

dense_store.write_documents(
    embedded_documents
)

print("✓ Dense DocumentStore created")
print(
    "Documents indexed:",
    dense_store.count_documents()
)

✓ Dense DocumentStore created
Documents indexed: 5


In [18]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=dense_store,
    top_k=3
)

print("✓ Dense Retriever created")

✓ Dense Retriever created


In [19]:
from haystack import component

@component
class DenseQueryEmbedder:

    @component.output_types(
        query_embedding=list[float]
    )
    def run(self, query: str):

        embedding = embedding_model.encode(
            query,
            convert_to_numpy=True
        )

        return {
            "query_embedding": embedding.tolist()
        }


dense_query_embedder = DenseQueryEmbedder()

dense_retriever_new = InMemoryEmbeddingRetriever(
    document_store=dense_store,
    top_k=3
)

dense_pipeline = Pipeline()

dense_pipeline.add_component(
    "query_embedder",
    dense_query_embedder
)

dense_pipeline.add_component(
    "retriever",
    dense_retriever_new
)

dense_pipeline.connect(
    "query_embedder.query_embedding",
    "retriever.query_embedding"
)

print("✓ Dense retrieval pipeline created")

✓ Dense retrieval pipeline created


In [20]:
dense_results = []

for i, question in enumerate(questions, start=1):

    result = dense_pipeline.run({
        "query_embedder": {
            "query": question
        }
    })

    retrieved_docs = result["retriever"]["documents"]

    dense_results.append({
        "question": question,
        "documents": retrieved_docs
    })

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print(question)

    for rank, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        print(f"\nRank {rank}")
        print("Score:", doc.score)
        print(
            "Text:",
            doc.content[:300].replace("\n", " ")
        )


QUESTION 1
What is the main objective of the research?

Rank 1
Score: 0.054528660930515484
Text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par

Rank 2
Score: 0.019213171606724574
Text: Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan Rewon Child Aditya Ramesh Daniel M. Ziegler Je

Rank 3
Score: 0.01706658950202477
Text: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim 

In [21]:
evaluation = []

print("Enter 1 if the TOP retrieved document is relevant.")
print("Enter 0 if the TOP retrieved document is not relevant.")

for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 70)
    print(f"Question {i}:")
    print(question)

    bm25_value = int(
        input("BM25 top result relevant? (1/0): ")
    )

    dense_value = int(
        input("Dense top result relevant? (1/0): ")
    )

    evaluation.append({
        "Question": question,
        "BM25": bm25_value,
        "Dense": dense_value
    })

print("\n✓ Manual evaluation completed!")

Enter 1 if the TOP retrieved document is relevant.
Enter 0 if the TOP retrieved document is not relevant.

Question 1:
What is the main objective of the research?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 2:
What problem does the research attempt to solve?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 3:
What methodology is used in the study?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 4:
What dataset is used?
BM25 top result relevant? (1/0): 0
Dense top result relevant? (1/0): 0

Question 5:
What model or algorithm is proposed?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 6:
How is the proposed method evaluated?
BM25 top result relevant? (1/0): 0
Dense top result relevant? (1/0): 1

Question 7:
What are the main findings of the study?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 0

Question 8:
What are the key experi

In [22]:
evaluation_df = pd.DataFrame(evaluation)

bm25_relevant = evaluation_df["BM25"].sum()
dense_relevant = evaluation_df["Dense"].sum()

bm25_precision = evaluation_df["BM25"].mean()
dense_precision = evaluation_df["Dense"].mean()

print(
    "BM25 relevant results:",
    bm25_relevant,
    "/ 10"
)

print(
    "Dense relevant results:",
    dense_relevant,
    "/ 10"
)

print(
    "\nBM25 precision:",
    round(bm25_precision * 100, 2),
    "%"
)

print(
    "Dense precision:",
    round(dense_precision * 100, 2),
    "%"
)

BM25 relevant results: 6 / 10
Dense relevant results: 7 / 10

BM25 precision: 60.0 %
Dense precision: 70.0 %


In [23]:
comparison = pd.DataFrame({
    "Method": [
        "BM25",
        "Dense Retrieval"
    ],
    "Questions": [
        10,
        10
    ],
    "Top-K": [
        3,
        3
    ],
    "Relevant Top Results": [
        bm25_relevant,
        dense_relevant
    ],
    "Precision (%)": [
        round(bm25_precision * 100, 2),
        round(dense_precision * 100, 2)
    ]
})

display(comparison)

,Method,Questions,Top-K,Relevant Top Results,Precision (%)
0,BM25,10,3,6,60.0
1,Dense Retrieval,10,3,7,70.0


In [24]:
display(evaluation_df)

,Question,BM25,Dense
0,What is the main objective of the research?,1,1
1,What problem does the research attempt to solve?,1,1
2,What methodology is used in the study?,1,1
3,What dataset is used?,0,0
4,What model or algorithm is proposed?,1,1
5,How is the proposed method evaluated?,0,1
6,What are the main findings of the study?,1,0
7,What are the key experimental results?,1,1
8,What are the limitations of the proposed appro...,0,1
9,What future work is suggested?,0,0


In [25]:
evaluation_df.to_csv(
    "haystack_retrieval_evaluation.csv",
    index=False
)

comparison.to_csv(
    "haystack_bm25_dense_comparison.csv",
    index=False
)

print("✓ Evaluation results saved")

✓ Evaluation results saved
